# Demo DocNLC trên GPU

Notebook này dùng để demo inference cho project **DocNLC: A Document Image Enhancement Framework with Normalized and Latent Contrastive Representation for Multiple Degradations**.

Ý nghĩa ngắn gọn của dự án: DocNLC khôi phục ảnh tài liệu bị suy giảm bởi nhiều loại nhiễu như blur, noise, shadow, watermark hoặc nền phức tạp. Model dùng kiến trúc kiểu U-Net (`SeeInDark`) kết hợp khối chuẩn hóa/contrast trong tầng đầu (`AttING`) để tăng cường ảnh tài liệu. Notebook này load checkpoint có sẵn, chạy ảnh demo bằng GPU, rồi lưu và hiển thị kết quả.

In [ ]:
!pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

## 1. Setup môi trường

Chạy cell này trước. Nếu môi trường GPU của bạn đã cài PyTorch CUDA thì notebook sẽ dùng luôn. Nếu chưa có PyTorch, hãy cài bản phù hợp với CUDA của máy theo lệnh được in ra.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_BASIC_DEPS = True

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def pip_install(*packages: str):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

if INSTALL_BASIC_DEPS:
    required = {
        "numpy": "numpy",
        "cv2": "opencv-python",
        "PIL": "pillow",
        "matplotlib": "matplotlib",
        "yaml": "pyyaml",
        "tqdm": "tqdm",
    }
    missing = [pkg for module, pkg in required.items() if not has_module(module)]
    if missing:
        pip_install(*missing)

if not has_module("torch"):
    print("PyTorch chưa được cài trong kernel này.")
    print("Gợi ý cài PyTorch GPU, ví dụ CUDA 12.1:")
    print(f"{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121")
    raise RuntimeError("Hãy cài PyTorch CUDA phù hợp với môi trường GPU rồi chạy lại notebook.")

print("Setup cơ bản đã sẵn sàng.")

## 2. Import thư viện và kiểm tra GPU

In [ ]:
from pathlib import Path
import os
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = torch.cuda.is_available()
device

## 3. Khai báo đường dẫn project, checkpoint và ảnh demo

Notebook này nên được đặt trong root project `DocNLC`. Nếu bạn chạy ở thư mục khác, chỉnh `PROJECT_ROOT` cho đúng.

In [ ]:
PROJECT_ROOT = Path.cwd()

# Nếu notebook không được mở từ root repo, tự tìm lên các thư mục cha.
if not (PROJECT_ROOT / "models" / "archs" / "EnhanceN_arch.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "models" / "archs" / "EnhanceN_arch.py").exists():
            PROJECT_ROOT = parent
            break

assert (PROJECT_ROOT / "models" / "archs" / "EnhanceN_arch.py").exists(), "Không tìm thấy root project DocNLC. Hãy chỉnh PROJECT_ROOT."

CKPT_PATH = PROJECT_ROOT / "ckpts" / "26_G.pth"
INPUT_IMAGE = PROJECT_ROOT / "TestImages" / "yeah.jpg"
REFERENCE_IMAGE = PROJECT_ROOT / "TestImages" / "yeah_processed.jpg"  # có thể dùng để so sánh nếu tồn tại
OUTPUT_DIR = PROJECT_ROOT / "demo_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

assert CKPT_PATH.exists(), f"Không tìm thấy checkpoint: {CKPT_PATH}"
assert INPUT_IMAGE.exists(), f"Không tìm thấy ảnh input: {INPUT_IMAGE}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CKPT_PATH:", CKPT_PATH)
print("INPUT_IMAGE:", INPUT_IMAGE)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 4. Load model DocNLC

Checkpoint trong repo khớp với generator `SeeInDark` trong `models/archs/EnhanceN_arch.py`. Cell này xử lý cả trường hợp checkpoint có/không có prefix `module.`.

In [ ]:
from models.archs.EnhanceN_arch import SeeInDark

def normalize_state_dict(raw_state):
    if isinstance(raw_state, dict) and "state_dict" in raw_state:
        raw_state = raw_state["state_dict"]
    if isinstance(raw_state, dict) and "params" in raw_state:
        raw_state = raw_state["params"]
    assert isinstance(raw_state, dict), "Checkpoint không phải state_dict hợp lệ."

    normalized = {}
    for key, value in raw_state.items():
        key = key.replace("module.", "", 1) if key.startswith("module.") else key
        normalized[key] = value
    return normalized

model = SeeInDark().to(device)
state = torch.load(CKPT_PATH, map_location="cpu")
state = normalize_state_dict(state)
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

print(f"Loaded checkpoint: {CKPT_PATH.name}")
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
if missing:
    print("Ví dụ missing:", missing[:5])
if unexpected:
    print("Ví dụ unexpected:", unexpected[:5])

## 5. Hàm tiền xử lý, chia patch, inference bằng GPU và ghép ảnh

In [ ]:
PATCH_SIZE = 256
THRESHOLD_OUTPUT = True
THRESHOLD_VALUE = 0.95

def read_bgr_float01(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(f"Không đọc được ảnh: {image_path}")
    if image.ndim == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    if image.shape[2] == 4:
        image = image[:, :, :3]
    return image.astype(np.float32) / 255.0

def resize_like_original_test(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    target_h, target_w = h, w
    if target_h < 384:
        target_h = 384
    if target_w < 384:
        target_w = 384
    while target_h % 4 != 0:
        target_h += 1
    while target_w % 4 != 0:
        target_w += 1
    if (target_h, target_w) != (h, w):
        image = cv2.resize(image, (target_w, target_h), interpolation=cv2.INTER_AREA)
    return image

def pad_to_patch_grid(image: np.ndarray, patch_size: int = PATCH_SIZE):
    h, w = image.shape[:2]
    padded_h = ((h // patch_size) + 1) * patch_size
    padded_w = ((w // patch_size) + 1) * patch_size
    padded = np.ones((padded_h, padded_w, 3), dtype=np.float32)
    padded[:h, :w] = image
    return padded, h, w

def run_docnlc_inference(image_path: Path, model: torch.nn.Module, device: torch.device, batch_size: int = 8):
    original_bgr = read_bgr_float01(image_path)
    test_image = resize_like_original_test(original_bgr)
    padded, valid_h, valid_w = pad_to_patch_grid(test_image, PATCH_SIZE)
    padded_h, padded_w = padded.shape[:2]

    patches = []
    positions = []
    for y in range(0, padded_h, PATCH_SIZE):
        for x in range(0, padded_w, PATCH_SIZE):
            patch = padded[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
            patches.append(patch)
            positions.append((y, x))

    output = np.zeros_like(padded, dtype=np.float32)
    start = time.time()
    with torch.no_grad():
        for i in tqdm(range(0, len(patches), batch_size), desc="Inference patches"):
            batch_np = np.stack(patches[i:i + batch_size], axis=0)
            batch = torch.from_numpy(batch_np).permute(0, 3, 1, 2).float().to(device, non_blocking=True)
            pred = model(batch)[0].clamp(0, 1).detach().cpu().permute(0, 2, 3, 1).numpy()
            for patch_pred, (y, x) in zip(pred, positions[i:i + batch_size]):
                output[y:y + PATCH_SIZE, x:x + PATCH_SIZE] = patch_pred

    output = output[:valid_h, :valid_w]
    if THRESHOLD_OUTPUT:
        output = (output > THRESHOLD_VALUE).astype(np.float32)

    output_uint8 = np.clip(output * 255.0, 0, 255).round().astype(np.uint8)
    elapsed = time.time() - start
    return original_bgr, output_uint8, elapsed

print("Hàm inference đã sẵn sàng.")

## 6. Chạy demo trên ảnh mẫu

In [ ]:
original_bgr, result_bgr, elapsed = run_docnlc_inference(INPUT_IMAGE, model, device, batch_size=8)

output_path = OUTPUT_DIR / f"{INPUT_IMAGE.stem}_docnlc.png"
cv2.imwrite(str(output_path), result_bgr)

print(f"Done in {elapsed:.2f}s on {device}.")
print("Saved:", output_path)

## 7. Hiển thị input, kết quả model và ảnh reference nếu có

In [ ]:
def bgr_to_rgb_uint8(image):
    if image.dtype != np.uint8:
        image = np.clip(image * 255.0, 0, 255).round().astype(np.uint8)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

images = [("Input degraded", bgr_to_rgb_uint8(original_bgr)), ("DocNLC output", bgr_to_rgb_uint8(result_bgr))]

if REFERENCE_IMAGE.exists():
    ref = cv2.imread(str(REFERENCE_IMAGE), cv2.IMREAD_UNCHANGED)
    if ref is not None:
        if ref.ndim == 2:
            ref = np.repeat(ref[:, :, None], 3, axis=2)
        images.append(("Reference in repo", bgr_to_rgb_uint8(ref)))

plt.figure(figsize=(6 * len(images), 8))
for idx, (title, image) in enumerate(images, 1):
    plt.subplot(1, len(images), idx)
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Chạy với ảnh khác

Đổi `CUSTOM_IMAGE` sang ảnh tài liệu bạn muốn demo. Kết quả sẽ được lưu trong `demo_outputs/`.

In [ ]:
CUSTOM_IMAGE = INPUT_IMAGE  # ví dụ: Path('/path/to/document.jpg')

custom_input_bgr, custom_result_bgr, custom_elapsed = run_docnlc_inference(Path(CUSTOM_IMAGE), model, device, batch_size=8)
custom_output_path = OUTPUT_DIR / f"{Path(CUSTOM_IMAGE).stem}_docnlc.png"
cv2.imwrite(str(custom_output_path), custom_result_bgr)

print(f"Done in {custom_elapsed:.2f}s on {device}.")
print("Saved:", custom_output_path)

plt.figure(figsize=(12, 8))
plt.subplot(1, 2, 1)
plt.imshow(bgr_to_rgb_uint8(custom_input_bgr))
plt.title("Input")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(bgr_to_rgb_uint8(custom_result_bgr))
plt.title("DocNLC output")
plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Ghi chú demo

- Checkpoint mặc định: `ckpts/26_G.pth`.
- Ảnh mẫu mặc định: `TestImages/yeah.jpg`.
- Output mặc định được threshold ở `0.95` giống logic trong `SIEN_model.test()` của repo. Nếu muốn xem ảnh continuous/raw hơn, đặt `THRESHOLD_OUTPUT = False` ở cell hàm inference rồi chạy lại các cell sau.
- Nếu ảnh rất lớn và GPU thiếu bộ nhớ, giảm `batch_size` trong hàm `run_docnlc_inference(...)`, ví dụ `batch_size=1` hoặc `2`.